In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_flim.tools_phasor as flim
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.dataio.mcs as mcs
from scipy.ndimage import shift
from skimage.registration import phase_cross_correlation


### Load the 4D (x,y,t,ch) h5 dataset of the G3BP1. Realignment of the time bins across the 25 channels was performed with the libttp library

In [ ]:
h = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\G3BP1_29_05_2024","r")
print(h.keys())
img = h["dataset_1"]
print(img.shape)


### Sum over the pixels ((x,y) dimensions of the datast) and display the TCSPC histograms along the channels

In [ ]:
data_hist = np.sum(img, axis=(0,1))
plt.figure()
for i in range(data_hist.shape[-1]):
    plt.plot(data_hist[:, i])

### Perform APR on each time bin to realign the 25 micro-images of the ISM dataset 

In [ ]:
import brighteyes_ism.analysis.APR_lib as apr
image_3d = np.sum(img, axis = -2)
print(image_3d.shape)
shift_vectors, err = apr.ShiftVectors(image_3d, usf = 100, ref = 12)
gr.PlotShiftVectors(shift_vectors)
print (shift_vectors) 

In [ ]:
from tqdm import tqdm
with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_G3BP1", 'w') as f:
     x_size, y_size, bin_size, channel_size = img.shape[0], img.shape[1], img.shape[2], img.shape[3]
# Create an empty dataset with dimensions (x,y,t, ch)
     dataset_shape = (x_size, y_size, bin_size, channel_size)
     h5_dataset = f.create_dataset('data', shape=dataset_shape, dtype=np.uint16)
    

     

     for bin in tqdm(range(img.shape[-2])):
         h5_dataset[:, :, bin, :] = apr.Reassignment(shift_vectors, img[:, :, bin, :], mode = 'interp')

### Calculate the intensity image (x,y) summing along the time and channels' dimensions ((t,ch) of the ISM dataset)

In [ ]:
h_apr = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_G3BP1","r")
print(h_apr.keys())
img_apr = h_apr["data"]
print(img_apr.shape)

In [ ]:
data_intensity = np.sum(img_apr, axis=(2,3)) 
print(data_intensity.shape)

### Show APR intensity image and compare it with the image of the central channel (closed confocal)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9,6))

gr.ShowImg(image_3d[:,:,12], pxsize_x = 0.146, fig = fig, ax = ax[0])
ax[0].set_title('Central pixel')

gr.ShowImg(data_intensity, pxsize_x = 0.146, fig = fig, ax = ax[1])
ax[1].set_title('APR')

fig.tight_layout()

### Plot the histograms of the number of photons in the X x Y pixels

In [ ]:
plt.figure()
plt.hist(data_intensity.flatten(), bins = 50 , range = (0, 50))

### Calculate the phasors for each pixel's TCSPC histogram of the ISM dataset and display the results in the phasor plot

In [ ]:
with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\APR_G3BP1","r") as f:
    
     print(f.keys())
     data_input = f["data"]  

     flim.phasor_h5(data_path = r"C:\Users\REPLACE_ME\Desktop\images\APR_G3BP1", data_input = data_input) 

In [ ]:
%matplotlib widget

hf_phasors_per_pixel = h5py.File( r"C:\Users\REPLACE_ME\Desktop\images\APR_G3BP1_phasors_matrix.h5", "r")
print(hf_phasors_per_pixel.keys())

phasors_pix = hf_phasors_per_pixel["h5_dataset_phasor_pix"]  # data with phasors in each pixel

print(phasors_pix.shape)

flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### Calculate the fluorescence lifetime from the phasor for each pixel with the formula below (f =  laser rep rate frequency):
### τφ = (1/(2πf)) * tan(φ)
### φ = arctan(s/g)
### g = Re{phasors_pix}
### s = Im{phasors_pix}

In [ ]:
tau_phi = flim.calculate_tau_phi(np.real(phasors_pix[:]), np.imag(phasors_pix[:]), dfd_freq = 80e6)
print(tau_phi.shape)

### Calculate the fluorescence lifetime from the phasor for each pixel with the formula below (f = dfd frequency or laser rep rate frequency):
### τ<sub>m</sub> = (1/2*π*f) * √(1/m<sup>2</sup> - 1)
### m = √g<sup>2</sup> + s<sup>2</sup>
### g = Re{phasor_corrected}
### s = Im{phasor_corrected}

In [ ]:
tau_m = flim.calculate_tau_m(np.real(phasors_pix), np.imag(phasors_pix), dfd_freq = 80e6)
print(tau_m.shape)

### Visualize histograms of tau distribution in the pixels

In [ ]:
tau_data = 1e9*tau_phi.flatten()

plt.figure()
plt.hist(tau_data, range = (-6, 10), bins = 50)

In [ ]:
tau_m_data = 1e9*tau_m.flatten()

plt.figure()
plt.hist(tau_m_data, range = (0, 13), bins = 50)

### Display and save the FLIM image representing the lifetime and intensity with a 2D colormap

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_intensity, tau_m*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 1000),
             fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\G3BP1_lifetime_and_phasor.pdf", dpi = 900)

### Comparison FLISM vs Closed Confocal FLIM (FLIM image on the 12th channel (central pixel of the SPAD array detector))

In [ ]:
flim.phasor_h5(data_path = r"C:\Users\REPLACE_ME\Desktop\images\G3BP1_29_05_2024", data_input = img[:, :, :, 12]) 

In [ ]:
%matplotlib widget

hf_phasors_per_pixel_12_channel = h5py.File( r"C:\Users\REPLACE_ME\Desktop\images\APR_G3BP1_phasors_matrix.h5", "r")
print(hf_phasors_per_pixel_12_channel.keys())

phasors_pix_12_ch = hf_phasors_per_pixel_12_channel["h5_dataset_phasor_pix"]  # data with phasors in each pixel for the 12th channel

print(phasors_pix_12_ch.shape)

flim.plot_phasor(phasors_pix_12_ch[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

### Lifetime calculation on the 12th channel's phasors

In [ ]:
tau_m_12_ch = flim.calculate_tau_m(np.real(phasors_pix_12_ch), np.imag(phasors_pix_12_ch), dfd_freq = 80e6)
print(tau_m_12_ch.shape)

In [ ]:
tau_m_data_12 = 1e9*tau_m_12_ch.flatten()

plt.figure()
plt.hist(tau_m_data_12, range = (0, 13), bins = 50)

In [ ]:
data_intensity_12_ch = np.sum(img[:, :, :, 12], axis = 2)
print(data_intensity_12_ch.shape)

In [ ]:
plt.figure()
plt.hist(data_intensity_12_ch.flatten(), bins = 5 , range = (0, 1000))

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
flim.plot_phasor(phasors_pix_12_ch[:], bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_intensity_12_ch, tau_m_12_ch*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 200),
             fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\G3BP1_lifetime_12_channel.pdf", dpi = 900)

### Comparing the FLIM image of the 12th channel and the FLIM image after performing APR on the detector's 25 channels

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9,6))

gr.show_flim(data_intensity_12_ch, tau_m_12_ch*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 200),
             fig = fig, ax = ax[0])  
ax[0].set_title('Closed Confocal FLIM')

gr.show_flim(data_intensity, tau_m*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 1000),
             fig = fig, ax = ax[1])
ax[1].set_title('APR-FLISM')

fig.tight_layout()

### Comparison FLISM vs Open Confocal FLIM (FLIM image on the sum of the 25 channels (without doing APR))

In [ ]:
data_sum = np.sum(img, axis=-1)
print(data_sum.shape)

### Calculate the phasors on the resulting image after summing on the channels

In [ ]:
flim.phasor_h5(data_path = r"C:\Users\REPLACE_ME\Desktop\images\Open_confocal\G3BP1_29_05_2024", data_input = data_sum) 

In [ ]:
%matplotlib widget

hf_phasors_per_pixel = h5py.File( r"C:\Users\REPLACE_ME\Desktop\images\Open_confocal\G3BP1_29_05_2024_phasors_matrix.h5", "r")
print(hf_phasors_per_pixel.keys())

phasors_pix_sum = hf_phasors_per_pixel["h5_dataset_phasor_pix"]  # data with phasors in each pixel

print(phasors_pix_sum.shape)

flim.plot_phasor(phasors_pix_sum[:], bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

In [ ]:
tau_m_sum = flim.calculate_tau_m(np.real(phasors_pix_sum), np.imag(phasors_pix_sum), dfd_freq = 80e6)
print(tau_m_sum.shape)

In [ ]:
tau_m_data_sum = 1e9*tau_m_sum.flatten()

plt.figure()
plt.hist(tau_m_data_sum, range = (0, 13), bins = 50)

In [ ]:
data_intensity_sum = np.sum(data_sum, axis = -1)
print(data_intensity_sum.shape)

In [ ]:
plt.figure()
plt.hist(data_intensity_sum.flatten(), bins = 5 , range = (0, 1000))

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
flim.plot_phasor(phasors_pix_sum[:], bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_intensity_sum, tau_m_sum*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 1000),
             fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\G3BP1_lifetime_open_confocal.pdf", dpi = 900)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9,6))

gr.show_flim(data_intensity_sum, tau_m_sum*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 1000),
             fig = fig, ax = ax[0])  
ax[0].set_title('Open Confocal FLIM')

gr.show_flim(data_intensity, tau_m*1e9, pxsize = 0.146, pxdwelltime =
             200, lifetime_bounds = (1.8, 3.6), intensity_bounds = (0, 1000),
             fig = fig, ax = ax[1])
ax[1].set_title('APR-FLISM')

fig.tight_layout()